In [1]:
# Carga del conjunto de datos

import pandas as pd

# Carga de datos
url = "https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv"
total_data = pd.read_csv(url)

# Visualización inicial
print(total_data.head())
print(total_data.info())

                                                 url  is_spam
0  https://briefingday.us8.list-manage.com/unsubs...     True
1                             https://www.hvper.com/     True
2                 https://briefingday.com/m/v4n3i4f3     True
3   https://briefingday.com/n/20200618/m#commentform    False
4                        https://briefingday.com/fan     True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2999 entries, 0 to 2998
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   url      2999 non-null   object
 1   is_spam  2999 non-null   bool  
dtypes: bool(1), object(1)
memory usage: 26.5+ KB
None


In [2]:
# Procesamiento de los enlaces

import regex as re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_url(url):
    # Eliminar protocolos y signos de puntuación, dejando solo texto
    url = re.sub(r'https?://', '', url)
    url = re.sub(r'[^a-zA-Z]', ' ', url)

    # Convertir a minúsculas y tokenizar
    tokens = url.lower().split()

    # Lematización y eliminación de stopwords (basado en image_884499.png)
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words("english"))

    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

# Aplicar el preprocesamiento
total_data["url_cleaned"] = total_data["url"].apply(preprocess_url)

# Transformar etiquetas (spam/ham) a numéricas si es necesario
# Supongamos que la columna es 'is_spam' (True/False o 1/0)
# total_data["is_spam"] = total_data["is_spam"].apply(lambda x: 1 if x == "spam" else 0)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
# 3 Division del conjunto y vectorización

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Vectorización
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(total_data["url_cleaned"])
y = total_data["is_spam"] # Ajustar según el nombre real de la columna

# División en train y test (image_8843fd.png)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# 4 Construir y Optimizar la SVM

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# SVM por defecto
model = SVC(kernel="linear", random_state=42)
model.fit(X_train, y_train)

# Optimización con Grid Search (Paso 4 de las instrucciones)
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': [0.1, 0.01]
}

grid = GridSearchCV(SVC(random_state=42), param_grid, cv=5)
grid.fit(X_train, y_train)

print(f"Mejores parámetros: {grid.best_params_}")
best_model = grid.best_estimator_

Mejores parámetros: {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}


In [5]:
import pickle

# Evaluación
y_pred = best_model.predict(X_test)
print(f"Accuracy final: {accuracy_score(y_test, y_pred)}")

# Guardado (image_8840d6.png)
filename = 'svm_url_spam_model_42.sav'
pickle.dump(best_model, open(filename, 'wb'))

Accuracy final: 0.96
